# exp_003 — LightGBM 2차 검증

LR에서 선별된 전처리가 비선형 모델에서도 유효한지 공용 LightGBM과 동일한 seed 42·5-Fold로 확인합니다. LightGBM 튜닝 실험이 아닙니다.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "common").is_dir()
        and (path / "experiments").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from experiments.SDH.exp_003_preprocessing.run_benchmark import run

print(f"프로젝트 루트: {PROJECT_ROOT}")

## 선별 후보 실행

- case 01: LightGBM 기준점
- case 06: 변이 유형별 개수
- case 10: hotspot top 50
- case 09: 최소 변이 빈도 10

In [ ]:
lgbm_cases = [
    "case_01_wt_binary",
    "case_06_mutation_types",
    "case_10_hotspot_top50",
    "case_09_min_count_10",
]

lgbm_leaderboard = run(
    selected_cases=lgbm_cases,
    model="lightgbm",
)
lgbm_leaderboard[
    [
        "preprocessing",
        "oof_f1_macro_mean",
        "oof_accuracy_mean",
        "fold_f1_macro_std",
        "elapsed_seconds",
    ]
]

## 선택적 3-seed 확인

위 결과에서 baseline보다 OOF Macro F1이 0.005 이상 높은 후보를 자동 선택합니다. 선택된 후보가 있으면 공정한 비교를 위해 baseline도 함께 seed 42/52/62로 재검증합니다.

In [ ]:
baseline_case = "case_01_wt_binary"
min_f1_improvement = 0.005

baseline_rows = lgbm_leaderboard.loc[
    lgbm_leaderboard["preprocessing"].eq(baseline_case)
]
if len(baseline_rows) != 1:
    raise RuntimeError("LightGBM baseline 결과를 하나만 찾을 수 있어야 합니다.")

baseline_f1 = float(baseline_rows.iloc[0]["oof_f1_macro_mean"])
selection_table = lgbm_leaderboard.copy()
selection_table["f1_improvement_vs_baseline"] = (
    selection_table["oof_f1_macro_mean"] - baseline_f1
)
confirmed_lgbm_cases = selection_table.loc[
    selection_table["preprocessing"].ne(baseline_case)
    & selection_table["f1_improvement_vs_baseline"].ge(
        min_f1_improvement
    ),
    "preprocessing",
].tolist()

display(
    selection_table[
        ["preprocessing", "oof_f1_macro_mean", "f1_improvement_vs_baseline"]
    ]
)
print(f"자동 선택 후보: {confirmed_lgbm_cases}")

if confirmed_lgbm_cases:
    confirmation_cases = [baseline_case, *confirmed_lgbm_cases]
    lgbm_confirmation = run(
        selected_cases=confirmation_cases,
        model="lightgbm",
        confirmation=True,
    )
    display(lgbm_confirmation)
else:
    print("baseline 대비 +0.005 이상인 후보가 없어 반복 검증을 건너뜁니다.")